### 1) Set up a local project workspace and Python virtual environment
Set up a local Python-based workspace of your choice (Databricks, Colab, Jupyter, etc.).
--> I will use a databricks notebook for the workspace

### 2) Ingest raw CSV and JSON datasets and preserve raw copies
Create storage space and connect it with your notebook.
--> I will generate fake data here

In [0]:
# import pandas as pd

# data = [
#     # Valid record
#     {"status": "active", "date": "2026-02-01", "monthly_fee": 29.99, "age": 25, "name": "Alice Smith", "user_id": "U001", "subcription": True, "address": "123 Main St", "email": "alice@example.com"},
#     # Invalid age (negative)
#     {"status": "inactive", "date": "2026-01-15", "monthly_fee": 19.99, "age": -5, "name": "Bob Jones", "user_id": "U002", "subcription": False, "address": "456 Oak Ave", "email": "bob@example.com"},
#     # Null user_id
#     {"status": "active", "date": "2026-02-03", "monthly_fee": 39.99, "age": 30, "name": "Carol Lee", "user_id": None, "subcription": True, "address": "789 Pine Rd", "email": "carol@example.com"},
#     # Duplicated user_id
#     {"status": "active", "date": "2026-02-04", "monthly_fee": 24.99, "age": 28, "name": "David Kim", "user_id": "U001", "subcription": True, "address": "321 Elm St", "email": "david@example.com"},
#     # Valid record
#     {"status": "inactive", "date": "2026-01-20", "monthly_fee": 15.99, "age": 22, "name": "Eva Brown", "user_id": "U003", "subcription": False, "address": "654 Maple Ave", "email": "eva@example.com"},
#     # Invalid age (string)
#     {"status": "active", "date": "2026-02-05", "monthly_fee": 29.99, "age": "twenty", "name": "Frank White", "user_id": "U004", "subcription": True, "address": "987 Cedar Rd", "email": "frank@example.com"},
#     # Null user_id
#     {"status": "inactive", "date": "2026-01-25", "monthly_fee": 19.99, "age": 35, "name": "Grace Black", "user_id": None, "subcription": False, "address": "159 Spruce St", "email": "grace@example.com"},
#     # Duplicated user_id
#     {"status": "active", "date": "2026-02-06", "monthly_fee": 34.99, "age": 40, "name": "Henry Green", "user_id": "U003", "subcription": True, "address": "753 Willow Ave", "email": "henry@example.com"},
#     # Valid record
#     {"status": "inactive", "date": "2026-01-30", "monthly_fee": 22.99, "age": 27, "name": "Ivy Blue", "user_id": "U005", "subcription": False, "address": "246 Birch Rd", "email": "ivy@example.com"},
#     # Invalid age (too high)
#     {"status": "active", "date": "2026-02-02", "monthly_fee": 29.99, "age": 200, "name": "Jack Red", "user_id": "U006", "subcription": True, "address": "369 Aspen St", "email": "jack@example.com"},
# ]

# df = pd.DataFrame(data)
# df.to_csv("/tmp/subscriptions.csv", index=False)

In [0]:
import pandas as pd

In [0]:
df = pd.read_csv('subscriptions.csv')

### 3) Clean and validate data using Pandas and NumPy
Standardise status to lowercase
Convert date columns to proper datetime
Replace missing monthly_fee with the median fee
Ensure age is numeric (invalid → NaN)
Drop records where user_id is missing or duplicated

In [0]:
df['status'] = df['status'].str.lower()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['monthly_fee'] = df['monthly_fee'].fillna(df['monthly_fee'].median())
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df = df.dropna(subset=['user_id'])
df = df[~df['user_id'].duplicated(keep=False)]

### 4) Engineer new analytical features and answer business questions
Create:

subscription_length_days
is_active_subscription (1/0)
revenue_estimate = monthly_fee × months_active

In [0]:
from datetime import datetime

# Calculate subscription_length_days
df['subscription_length_days'] = (datetime(2026, 2, 6) - df['date']).dt.days

# Engineer is_active_subscription (1/0)
df['is_active_subscription'] = (df['status'] == 'active').astype(int)

# Calculate months_active (rounded up)
df['months_active'] = (df['subscription_length_days'] / 30).apply(lambda x: int(x) + (x % 1 > 0))

# Calculate revenue_estimate
df['revenue_estimate'] = df['monthly_fee'] * df['months_active']

### 5) Integrate an external public API with error handling

In [0]:
import requests

# Example: Get latest exchange rates from exchangerate-api.com
response = requests.get("https://open.er-api.com/v6/latest/USD")
exchange_data = response.json()

In [0]:
# Fetch country names for country codes using restcountries.com
country_codes = list(exchange_data['rates'].keys())

response = requests.get(f"https://restcountries.com/v3.1/alpha?codes={','.join(country_codes)}")
countries_info = response.json()

country_names = {c['cca2']: c['name']['common'] for c in countries_info if 'cca2' in c and 'name' in c}

In [0]:
# Add country_code column to df based on email domain (example logic)
df['country_code'] = df['email'].str.split('.').str[-1].str.upper()

# Map country names to df using country_names dict
df['country_name'] = df['country_code'].map(country_names)

# Map exchange rates to df using exchange_data['rates']
df['exchange_rate'] = df['country_code'].map(exchange_data['rates'])

display(df)

### 6) Load cleaned data into a relational database
--> I choose to create managed tables in databricks instead

In [0]:
import pyspark.pandas as ps

# Convert pandas df to Spark DataFrame
sdf = spark.createDataFrame(df)

# Create users table
users_df = sdf.select(
    "user_id", "name", "age", "email", "address", "country_code", "country_name"
)
users_df.write.mode("overwrite").saveAsTable("users")

# Create subscriptions table
subscriptions_df = sdf.select(
    "user_id", "status", "date", "monthly_fee", "subcription", "is_active_subscription", "months_active", "revenue_estimate", "exchange_rate"
)
subscriptions_df.write.mode("overwrite").saveAsTable("subscriptions")

# Create analytics_summary table
analytics_summary_df = sdf.groupBy("country_code", "country_name").agg(
    {"user_id": "count", "revenue_estimate": "sum", "is_active_subscription": "sum"}
).withColumnRenamed("count(user_id)", "user_count") \
 .withColumnRenamed("sum(revenue_estimate)", "total_revenue") \
 .withColumnRenamed("sum(is_active_subscription)", "active_subscriptions")
analytics_summary_df.write.mode("overwrite").saveAsTable("analytics_summary")

In [0]:
spark.sql("""
    select country_code, user_count, total_revenue, active_subscriptions from analytics_summary
""").show()

### 7) Use Git for version control

I did

### Questions to Answer

1. When is it better to drop records vs. impute values, and how does this depend on business context?
--> When to drop records vs. impute values depends on the importance of the missing data and the business impact. Drop records when missing or invalid data makes them unusable for analysis (e.g., missing user_id, which is a unique identifier), or when the proportion of such records is small and won’t bias results. Impute values when the missing data is not critical, and you can reasonably estimate it (e.g., filling missing monthly_fee with the median). The choice depends on whether data completeness or accuracy is more critical for your business goals.


2. What assumptions are you making when estimating revenue this way, and how could they bias conclusions?
--> Estimating revenue as monthly_fee × months_active assumes all users pay the full fee every month and remain subscribed for the entire period. This ignores churn, discounts, failed payments, or partial months. Such assumptions can overestimate revenue and bias conclusions, especially if many users cancel early or have variable billing.

3. Why should API failures not crash a data pipeline?
--> API failures should not crash a data pipeline because external services can be unreliable. If a pipeline fails on every API error, it can halt business operations and delay data availability. Instead, handle errors gracefully (e.g., retry, log, or skip) to ensure the pipeline continues running and delivers as much data as possible.

4. Why is schema design just as important as data cleaning?
--> Schema design is as important as data cleaning because a well-designed schema enforces data integrity, consistency, and usability. It defines data types, relationships, and constraints, preventing many data quality issues before they occur. Good schema design makes downstream analysis, integration, and maintenance easier and more reliable.


5. How does version control improve trust in data outputs?
--> Version control improves trust in data outputs by tracking changes to code, schemas, and transformation logic. It enables reproducibility, auditing, and rollback to previous states. This transparency ensures that data outputs can be traced to specific code versions, making it easier to identify, review, and fix issues.